# JEV decision model: inference demo

Loads a checkpoint written by `train.py` and shows the raw API (`choice`, `noul`, `score`), the JEV-style `decide` API, and the batched forms.

Set `CHECKPOINT` below to a `best_val` directory (or the output directory containing it).

In [1]:
import json
import sys
import time
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "jevbench").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from jevbench.inference import JevDecider

CHECKPOINT = "leobitz/decision-master-base-label-encode-0.1"  # or checkpoints_jev_bev
decider = JevDecider.from_checkpoint(CHECKPOINT, batch_size=32)
print("architecture:", decider.args.architecture, "| device:", decider.device)


def show(obj):
    print(json.dumps(obj, indent=2))

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

model.pt:   0%|          | 0.00/1.29G [00:00<?, ?B/s]

The tokenizer you are loading from '/home/leo/.cache/huggingface/hub/models--leobitz--decision-master-base-label-encode-0.1/snapshots/caeced6494aad3efbd5e29c9b6dfd7d12bf33f79' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

architecture: hierarchical | device: cuda


## Choice

Options can be a plain list or a `{key: description}` mapping. Include a catch-all such as `needs_review`.

In [2]:
context = "My card was charged twice for the same invoice last week."
query = "Which team should handle this request?"
choices = {
    "billing": "Payments, invoices, refunds, or payouts",
    "technical": "Bugs, outages, or integration failures",
    "sales": "Pricing, upgrades, or new accounts",
    "needs_review": "The request does not clearly fit another option",
}

show(decider.choice(context, query, choices))

{
  "type": "choice",
  "choice": "needs_review",
  "probabilities": {
    "billing": 0.006898073627596345,
    "technical": 0.00036558451007562243,
    "sales": 0.00046941979773739356,
    "needs_review": 0.9922669220645907
  },
  "confidence": 0.9922669220645907
}


## Noul (yes/no)

`noul` is P(yes). Internally the question is asked with every yes/no wording seen in training (yes/no, true/false, correct/incorrect, ...) in both orders, and the results are averaged.

In [3]:
show(decider.noul(context, "Does this message explicitly request a refund?"))
show(decider.noul("What is your pricing for 50 seats?", "Does this message explicitly request a refund?"))

{
  "type": "noul",
  "noul": 0.04627660803629518
}
{
  "type": "noul",
  "noul": 0.008193320083387577
}


## Score

Levels are ordered low to high. The score is probability-weighted, so it can fall between levels (numeric keys are used as values, otherwise 0-based indices).

In [4]:
levels = {
    "routine": "No time pressure; can wait for normal handling",
    "elevated": "Blocks a user or needs same-day attention",
    "critical": "Outage or data loss affecting many users",
}

show(decider.score("The whole site has been down for all customers since 9am.", "How urgent is this request?", levels))

{
  "type": "score",
  "score": 1.599961863599428,
  "legend": {
    "routine": "No time pressure; can wait for normal handling",
    "elevated": "Blocks a user or needs same-day attention",
    "critical": "Outage or data loss affecting many users"
  },
  "probabilities": {
    "routine": 0.18570010981944143,
    "elevated": 0.02863791676168939,
    "critical": 0.7856619734188692
  },
  "confidence": 0.7856619734188692
}


## JEV-style `decide`

One shared state, several independent questions. The state can be text, a JSON object, or a list.

In [5]:
state = {
    "ticket": "My card was charged twice for the same invoice last week.",
    "account_tier": "enterprise",
}
questions = {
    "department": {"type": "choice", "instructions": "Which team should handle this request?", "criteria": choices},
    "urgency": {"type": "score", "instructions": "How urgent is this request?", "criteria": levels},
    "refund": {"type": "noul", "instructions": "Does this message explicitly request a refund?"},
}

result = decider.decide(state, questions)
show(result)

{
  "answers": {
    "department": {
      "type": "choice",
      "choice": "needs_review",
      "probabilities": {
        "billing": 0.007106023223558179,
        "technical": 0.0008486935222392302,
        "sales": 0.0013144822744012047,
        "needs_review": 0.9907308009798014
      },
      "confidence": 0.9907308009798014
    },
    "urgency": {
      "type": "score",
      "score": 0.06720087512192846,
      "legend": {
        "routine": "No time pressure; can wait for normal handling",
        "elevated": "Blocks a user or needs same-day attention",
        "critical": "Outage or data loss affecting many users"
      },
      "probabilities": {
        "routine": 0.9456186114802051,
        "elevated": 0.04156190191766154,
        "critical": 0.01281948660213346
      },
      "confidence": 0.9456186114802051
    },
    "refund": {
      "type": "noul",
      "noul": 0.21875897520077103
    }
  }
}


### Policy stays in application code

Thresholds below are placeholders; choose real ones from a labeled evaluation set.

In [6]:
ALLOWED = set(choices) - {"needs_review"}
dept = result["answers"]["department"]

if dept["choice"] in ALLOWED and dept["confidence"] >= 0.7:
    print("route to:", dept["choice"])
else:
    print("send to human review")

if result["answers"]["refund"]["noul"] >= 0.5:
    print("flag: refund requested (still check authorization before acting)")

send to human review


## Batched forms

Items are dicts. All variants of all items are packed into shared model batches; answers come back in input order.

In [7]:
tickets = [
    "My card was charged twice for the same invoice last week.",
    "The API returns a 500 error whenever I upload a file.",
    "Can I upgrade from the starter plan to the team plan?",
]

routed = decider.choice_batch([{"context": t, "query": "Which team should handle this request?", "choices": choices} for t in tickets])
refunds = decider.noul_batch([{"context": t, "query": "Does this message explicitly request a refund?"} for t in tickets])

for t, r, n in zip(tickets, routed, refunds):
    print(f"{r['choice']:<13} conf={r['confidence']:.2f}  refund={n['noul']:.2f}  | {t}")

needs_review  conf=0.99  refund=0.05  | My card was charged twice for the same invoice last week.
needs_review  conf=0.96  refund=0.00  | The API returns a 500 error whenever I upload a file.
needs_review  conf=0.94  refund=0.05  | Can I upgrade from the starter plan to the team plan?


In [8]:
# decide_batch: a different state per request, each with its own questions.
requests = [{"state": t, "questions": questions} for t in tickets]
answers = decider.decide_batch(requests)

for t, a in zip(tickets, answers):
    ans = a["answers"]
    print(f"{ans['department']['choice']:<13} urgency={ans['urgency']['score']:.2f}  refund={ans['refund']['noul']:.2f}  | {t}")

needs_review  urgency=0.05  refund=0.05  | My card was charged twice for the same invoice last week.
needs_review  urgency=0.08  refund=0.00  | The API returns a 500 error whenever I upload a file.
needs_review  urgency=0.03  refund=0.05  | Can I upgrade from the starter plan to the team plan?


### Looping vs batching

In [9]:
items = [{"context": t, "query": "Does this message explicitly request a refund?"} for t in tickets * 4]

start = time.perf_counter()
looped = [decider.noul(i["context"], i["query"]) for i in items]
t_loop = time.perf_counter() - start

start = time.perf_counter()
batched = decider.noul_batch(items)
t_batch = time.perf_counter() - start

print(f"loop: {t_loop:.2f}s | batch: {t_batch:.2f}s | {len(items)} items")
print("max |difference|:", max(abs(a["noul"] - b["noul"]) for a, b in zip(looped, batched)))

loop: 0.77s | batch: 0.26s | 12 items
max |difference|: 0.0062643135857470555
